# 06 — Linear Forecasting Models (OLS, Ridge, Lasso)

## 1. Introduction and model specification

**Task.** As in Notebook 05: at 12:00 on day $t$, we forecast total electricity consumption in Basel-Stadt on day $d = t + 1$ (MWh/day), using only information available at that time. Following the professor-approved assumption, consumption is treated as known only from day $d-5$ and older (`lag_5` and older).

**Why linear regression?** The lag-7 baseline (Notebook 05) uses a single past value. A linear regression combines several pieces of information available at the cutoff:

- older consumption (`lag_5`, `lag_6`, `lag_7`, `lag_14`, `lag_21`, `lag_28` and the weekly average `lag_mean_5_11`);
- archived weather forecasts for the target day ($W_d$);
- calendar information, including holidays on the target day and on the lagged days ($C_d$).

**Model specification.**

$$
\hat y_d = \beta_0 + \sum_{k \in \{5,6,7,14,21,28\}} \beta_k\, y_{d-k} + \beta_{m}\, \bar y_{d-11:d-5} + \gamma^\top W_d + \delta^\top C_d
$$

Because past values of the target enter as predictors alongside external variables, this is an autoregressive model with exogenous variables (ARX-type). It is not an unrestricted autoregressive model: lags 1–4 are deliberately excluded because they would not be available at the forecasting cutoff.

**Three estimators.**
- **OLS** chooses the coefficients that minimise the sum of squared prediction errors on the training data.
- **Ridge** adds an L2 penalty $\alpha \sum_j \beta_j^2$. It shrinks coefficients towards zero, which stabilises the estimates when predictors are strongly correlated, as our consumption lags are.
- **Lasso** adds an L1 penalty $\alpha \sum_j |\beta_j|$. It can set some coefficients exactly to zero and thereby selects a subset of predictors.

Both penalties depend on the scale of each predictor, so continuous predictors are standardised before fitting. The penalty strength $\alpha$ is chosen by time-ordered cross-validation within each training window (Section 6).

**Evaluation.** All models are evaluated in the 12 monthly validation folds of 2025, on exactly the same 365 target dates as the baseline, using MAE (primary) and RMSE (secondary) in MWh/day. The 2026 holdout is not used in this notebook.

In [1]:
# Setup: imports, project paths and methodology constants
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A Lasso that has not converged must never go unnoticed: turn the warning into an error
warnings.simplefilter("error", category=ConvergenceWarning)


def find_repo_root(start: Path) -> Path:
    """Walk upwards from `start` until the folder containing data/processed/features.csv is found."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "processed" / "features.csv").exists():
            return folder
    raise FileNotFoundError(
        f"data/processed/features.csv not found above {start}. "
        "Check that you opened the repository, or run code/04_prepare_features.py first."
    )


ROOT = find_repo_root(Path.cwd().resolve())
FEATURES_PATH = ROOT / "data" / "processed" / "features.csv"
FIG_DIR = ROOT / "output" / "figures"
TABLE_DIR = ROOT / "output" / "tables"
BASELINE_PREDICTIONS_PATH = TABLE_DIR / "baseline_validation_predictions.csv"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Methodology constants (identical to Notebook 05 and the approved methodology)
PERIODS = {
    "train": ("2018-01-01", "2024-12-31"),
    "validation": ("2025-01-01", "2025-12-31"),
    "test": ("2026-01-01", "2026-10-31"),
}
MIN_LAG = 5                                   # youngest consumption lag allowed at the cutoff
TRAIN_START = pd.Timestamp(PERIODS["train"][0])
VALIDATION_MONTHS = pd.period_range("2025-01", "2025-12", freq="M")  # the 12 monthly folds

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("Repository root:", ROOT)

Python: 3.11.9
pandas: 3.1.1 | numpy: 2.4.6 | scikit-learn: 1.9.1
Repository root: /Users/mariegrandfils/Library/CloudStorage/OneDrive-Personal/HSG/Bachelor Economics/DSF/DSF_Group_Project/DSF_forecasting_electricity_consumption_Basel


## 2. Load and check the data

We load `data/processed/features.csv` (prepared by Person 1, not rebuilt here) and repeat the essential structural checks from Notebook 05. We also load the baseline predictions exported by Notebook 05, which are needed for the comparison in Section 9.

In [2]:
# Load the shared feature table (no downloading, no rebuilding) and check its structure
df = pd.read_csv(FEATURES_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

EXPECTED_COLUMNS = [
    "date", "period", "consumption_mwh", "baseline", "baseline_uses_fallback",
    "weekday", "is_weekend", "month", "is_holiday", "is_christmas_period",
    "lag_7_is_holiday", "lag_14_is_holiday",
    "forecast_temp_mean", "forecast_temp_min", "forecast_temp_max",
    "forecast_precipitation", "heating_degrees", "cooling_degrees",
    "lag_5", "lag_6", "lag_7", "lag_14", "lag_21", "lag_28", "lag_mean_5_11",
]
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

lag_numbers = [int(c[4:]) for c in df.columns if c.startswith("lag_") and c[4:].isdigit()]
assert min(lag_numbers) >= MIN_LAG, f"Lag younger than lag_{MIN_LAG} found"

assert df["date"].is_unique, "Duplicate target dates"
full_calendar = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
assert len(full_calendar.difference(df["date"])) == 0, "Missing calendar days"

for name, (start, end) in PERIODS.items():
    in_window = df["date"].between(pd.Timestamp(start), pd.Timestamp(end))
    assert (df.loc[in_window, "period"] == name).all(), f"Period labels do not match the {name} window"

assert df.isna().sum().sum() == 0, "Missing values found: decide on a treatment before modelling"

print("Shape (rows, columns):", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
display(df.groupby("period", sort=False)["date"].agg(first="min", last="max", days="count"))
print("All structural checks passed: columns, lags >= lag_5, unique dates, no gaps, periods, no missing values")

# Baseline predictions from Notebook 05 (needed for the comparison in Section 9)
baseline_pred = pd.read_csv(BASELINE_PREDICTIONS_PATH, parse_dates=["date"])
validation_dates = df.loc[df["period"] == "validation", "date"]
assert len(baseline_pred) == len(validation_dates) == 365, "Baseline file does not have 365 rows"
assert baseline_pred["date"].isin(validation_dates).all(), "Baseline dates differ from the 2025 validation dates"
print(f"Baseline predictions loaded: {len(baseline_pred)} days, "
      f"MAE = {baseline_pred['absolute_error_mwh'].mean():.1f} MWh/day (Notebook 05: 137.3)")

Shape (rows, columns): (3226, 25)
Date range: 2018-01-01 to 2026-10-31


,first,last,days
period,,,
train,2018-01-01,2024-12-31,2557
validation,2025-01-01,2025-12-31,365
test,2026-01-01,2026-10-31,304


All structural checks passed: columns, lags >= lag_5, unique dates, no gaps, periods, no missing values
Baseline predictions loaded: 365 days, MAE = 137.3 MWh/day (Notebook 05: 137.3)


## 3. Choice of explanatory variables

All predictors come from `features.csv` (Person 1); none are created here.

| Group | Columns | Treatment |
|---|---|---|
| Consumption lags | `lag_5`, `lag_6`, `lag_7`, `lag_14`, `lag_21`, `lag_28`, `lag_mean_5_11` | numeric, standardised |
| Weather forecasts | `heating_degrees`, `cooling_degrees`, `forecast_precipitation` | numeric, standardised |
| Calendar (categorical) | `weekday`, `month` | one-hot encoded |
| Calendar (binary) | `is_holiday`, `is_christmas_period`, `lag_7_is_holiday`, `lag_14_is_holiday` | kept as 0/1 |

**Never used as predictors:** `date`, `period`, `consumption_mwh` (the target), and `baseline` / `baseline_uses_fallback`. The baseline is reserved for benchmarking; besides, it equals `lag_7`, which is already included.

**Columns left out to avoid redundant information (verified below):**
- `forecast_temp_mean`: by construction, `heating_degrees − cooling_degrees = 15 − forecast_temp_mean` on every day. With an intercept, OLS cannot estimate separate coefficients for all three (perfect multicollinearity). We keep heating and cooling degrees, because they represent the U-shaped temperature effect found in the exploratory analysis.
- `forecast_temp_min`, `forecast_temp_max`: correlated at about 0.97 with the mean temperature, so they add little information but make individual coefficients unstable.
- `is_weekend`: once `weekday` is one-hot encoded, `is_weekend` equals the sum of the Saturday and Sunday indicators (dummy variable trap).

**Two specifications, fixed in advance.** To see whether the weather forecasts improve predictions, every model is estimated twice:
- `no_weather`: consumption lags + calendar;
- `with_weather`: the same plus the three weather variables.

No further specifications are tried, so model selection stays limited and transparent.

In [3]:
# Feature groups (all prepared by Person 1 in 04_prepare_features.py)
TARGET = "consumption_mwh"
LAG_FEATURES = ["lag_5", "lag_6", "lag_7", "lag_14", "lag_21", "lag_28", "lag_mean_5_11"]
WEATHER_FEATURES = ["heating_degrees", "cooling_degrees", "forecast_precipitation"]
CATEGORICAL_FEATURES = ["weekday", "month"]   # one-hot encoded in Section 4
BINARY_FEATURES = ["is_holiday", "is_christmas_period", "lag_7_is_holiday", "lag_14_is_holiday"]
NEVER_PREDICTORS = ["date", "period", TARGET, "baseline", "baseline_uses_fallback"]

# The two specifications compared on the 2025 validation data (fixed in advance)
SPECS = {
    "no_weather": {"numeric": LAG_FEATURES,
                   "categorical": CATEGORICAL_FEATURES, "binary": BINARY_FEATURES},
    "with_weather": {"numeric": LAG_FEATURES + WEATHER_FEATURES,
                     "categorical": CATEGORICAL_FEATURES, "binary": BINARY_FEATURES},
}


def spec_columns(spec):
    """All input columns of a specification, in a fixed order."""
    return spec["numeric"] + spec["categorical"] + spec["binary"]


# Checks: no forbidden column, no duplicates, no consumption lag younger than lag_5
for name, spec in SPECS.items():
    cols = spec_columns(spec)
    assert not set(cols) & set(NEVER_PREDICTORS), f"'{name}' contains a column that must not be a predictor"
    assert len(cols) == len(set(cols)), f"'{name}' lists a column twice"
    lags = [int(c[4:]) for c in cols if c.startswith("lag_") and c[4:].isdigit()]
    assert min(lags) >= MIN_LAG, f"'{name}' uses a lag younger than lag_{MIN_LAG}"
    print(f"Specification '{name}': {len(cols)} input columns")

# Why some columns are left out: verified on the data
BASE_TEMP = 15.0   # heating/cooling threshold fixed in 04_prepare_features.py
identity_gap = ((df["heating_degrees"] - df["cooling_degrees"]) - (BASE_TEMP - df["forecast_temp_mean"])).abs().max()
print(f"\nheating_degrees − cooling_degrees vs. 15 − forecast_temp_mean: largest difference {identity_gap:.1e} -> exact identity")
weekend_from_weekday = df["weekday"].isin([5, 6]).astype(int)
print("is_weekend equals the Saturday + Sunday dummies on every row:", (weekend_from_weekday == df["is_weekend"]).all())
print("\nCorrelation of the temperature forecasts (training period 2018–2024):")
display(df.loc[df["period"] == "train", ["forecast_temp_mean", "forecast_temp_min", "forecast_temp_max"]].corr().round(2))

Specification 'no_weather': 13 input columns
Specification 'with_weather': 16 input columns

heating_degrees − cooling_degrees vs. 15 − forecast_temp_mean: largest difference 0.0e+00 -> exact identity
is_weekend equals the Saturday + Sunday dummies on every row: True

Correlation of the temperature forecasts (training period 2018–2024):


,forecast_temp_mean,forecast_temp_min,forecast_temp_max
forecast_temp_mean,1.00,0.97,0.97
forecast_temp_min,0.97,1.00,0.89
forecast_temp_max,0.97,0.89,1.00


## 4. Preprocessing

**Two kinds of transformations.**
- *Deterministic feature engineering* (Person 1, `04_prepare_features.py`): lags, holiday flags and heating/cooling degrees follow fixed formulas. Nothing is estimated from the data, so they cannot leak information.
- *Learned preprocessing* (here): imputation medians, standardisation means and standard deviations, and the categories of the one-hot encoder are estimated from data. They must be estimated **only from the training rows of each fold**; otherwise information from the validation period leaks into training.

**Steps, combined in a scikit-learn `Pipeline`:**
- Numeric columns: `SimpleImputer` (median) → `StandardScaler`. There are no missing values in `features.csv` (Section 2); the imputer is a safeguard.
- `weekday` and `month`: `OneHotEncoder(drop="first")`. The first category is dropped to avoid the dummy variable trap with the intercept, so Monday and January are the reference categories. Weekday 0–6 and month 1–12 are categories, not quantities: Sunday (6) is not "six times" Monday.
- Binary flags: kept unchanged as 0/1.

Because the preprocessing and the regression model form one `Pipeline`, calling `fit` on a fold's training rows learns all parameters from those rows only; `predict` then applies them unchanged to the validation month. The same preprocessing is used for OLS, Ridge and Lasso, so the models differ only in their estimator.

In [4]:
def make_preprocessor(spec):
    """Learned preprocessing. Inside a Pipeline it is fitted on the training rows of a fold only."""
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),   # safeguard: features.csv has no missing values
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("numeric", numeric_steps, spec["numeric"]),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
         spec["categorical"]),
        ("binary", "passthrough", spec["binary"]),
    ])


def make_pipeline(model_name, spec):
    """Preprocessing + regression model in one object: fit() learns everything from the training rows only."""
    models = {
        "ols": LinearRegression(),
        "ridge": Ridge(),
        "lasso": Lasso(max_iter=100_000),
    }
    return Pipeline([("preprocess", make_preprocessor(spec)), ("model", models[model_name])])


# Demonstration on the January 2025 fold: what does the preprocessing learn, and from which rows?
demo_cutoff = VALIDATION_MONTHS[0].start_time - pd.Timedelta(days=MIN_LAG)
demo_train = df[(df["date"] >= TRAIN_START) & (df["date"] <= demo_cutoff)]
demo_spec = SPECS["with_weather"]
demo_pre = make_preprocessor(demo_spec).fit(demo_train[spec_columns(demo_spec)])

feature_names = demo_pre.get_feature_names_out()
print(f"Training rows of the January fold: {len(demo_train)} "
      f"({demo_train['date'].min().date()} to {demo_train['date'].max().date()})")
print("Columns after preprocessing:", len(feature_names))
print(list(feature_names))

scaler = demo_pre.named_transformers_["numeric"].named_steps["scale"]
assert np.isclose(scaler.mean_[0], demo_train["lag_5"].mean())
print(f"\nMean of lag_5 learned by the scaler: {scaler.mean_[0]:.1f} MWh/day "
      f"= mean of lag_5 in the training rows of this fold only")

Training rows of the January fold: 2553 (2018-01-01 to 2024-12-27)
Columns after preprocessing: 31
['numeric__lag_5', 'numeric__lag_6', 'numeric__lag_7', 'numeric__lag_14', 'numeric__lag_21', 'numeric__lag_28', 'numeric__lag_mean_5_11', 'numeric__heating_degrees', 'numeric__cooling_degrees', 'numeric__forecast_precipitation', 'categorical__weekday_1', 'categorical__weekday_2', 'categorical__weekday_3', 'categorical__weekday_4', 'categorical__weekday_5', 'categorical__weekday_6', 'categorical__month_2', 'categorical__month_3', 'categorical__month_4', 'categorical__month_5', 'categorical__month_6', 'categorical__month_7', 'categorical__month_8', 'categorical__month_9', 'categorical__month_10', 'categorical__month_11', 'categorical__month_12', 'binary__is_holiday', 'binary__is_christmas_period', 'binary__lag_7_is_holiday', 'binary__lag_14_is_holiday']

Mean of lag_5 learned by the scaler: 3463.8 MWh/day = mean of lag_5 in the training rows of this fold only


## 5. Ordinary least squares (OLS)

**Monthly folds and the availability cutoff.** Each validation month is one fold. The model for a month is fitted once, at 12:00 on the last day of the previous month, when the first forecast of that month is issued. At that moment, consumption is known only up to five days before the first target day. Since every training row needs its observed target, a fold's training rows are all target dates from 1 January 2018 up to **(first day of the month − 5 days)**. For example, the January 2025 fold trains on targets up to 27 December 2024, and the February fold on targets up to 27 January 2025. All days of the validation month are then predicted with this fixed model, using each day's own features, which are available at that day's cutoff.

A simpler rule such as "all dates before the month" would train on targets that are not yet published at the refit time. The function `get_fold` implements the correct rule and checks it.

**OLS.** `LinearRegression` estimates the coefficients by minimising the sum of squared errors on the training rows. As a first check before running all 12 folds (Section 8), we fit it on the January fold: predictions must be finite and aligned with the validation dates. We also contrast the *in-sample* error (on the rows the model was fitted on) with the *out-of-sample* error (on the unseen validation month). Only the latter measures forecast accuracy.

In [8]:
def compute_metrics(actual, predicted):
    """Number of days, MAE, RMSE and mean error in MWh/day (same function as Notebook 05)."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    assert len(actual) == len(predicted) > 0, "Empty or misaligned inputs"
    assert np.isfinite(actual).all() and np.isfinite(predicted).all(), "Missing or non-finite values"
    return {
        "n_days": len(actual),
        "mae_mwh": mean_absolute_error(actual, predicted),
        "rmse_mwh": root_mean_squared_error(actual, predicted),
        "mean_error_mwh": np.mean(actual - predicted),
    }


def get_fold(month):
    """Training and validation rows of one monthly fold, respecting the 5-day availability rule."""
    first_day = month.start_time
    last_day = month.end_time.normalize()
    cutoff = first_day - pd.Timedelta(days=MIN_LAG)   # newest target date known at the refit
    train = df[(df["date"] >= TRAIN_START) & (df["date"] <= cutoff)]
    valid = df[(df["date"] >= first_day) & (df["date"] <= last_day)]
    assert train["date"].is_monotonic_increasing, "Training rows must be in time order"
    assert (train["period"] != "test").all() and (valid["period"] == "validation").all()
    assert train["date"].max() <= cutoff < valid["date"].min()
    return train, valid, cutoff


# Demonstration: OLS on the first fold (January 2025), specification with weather
demo_month = VALIDATION_MONTHS[0]
train, valid, cutoff = get_fold(demo_month)
spec = SPECS["with_weather"]
cols = spec_columns(spec)

ols = make_pipeline("ols", spec).fit(train[cols], train[TARGET])
pred = ols.predict(valid[cols])
assert len(pred) == len(valid) and np.isfinite(pred).all(), "Predictions missing or not finite"

print(f"Fold {demo_month}: trained on {len(train)} days ({train['date'].min().date()} to {cutoff.date()}), "
      f"predicting {len(valid)} days ({valid['date'].min().date()} to {valid['date'].max().date()})")
in_sample = compute_metrics(train[TARGET], ols.predict(train[cols]))["mae_mwh"]
out_of_sample = compute_metrics(valid[TARGET], pred)["mae_mwh"]
print(f"In-sample MAE on the training rows: {in_sample:.1f} MWh/day  (NOT a forecast accuracy)")
print(f"Out-of-sample MAE, January 2025:    {out_of_sample:.1f} MWh/day")
display(pd.DataFrame({"date": valid["date"].values, "actual_mwh": valid[TARGET].values,
                      "predicted_mwh": pred}).head().round({"actual_mwh": 1, "predicted_mwh": 1}))

Fold 2025-01: trained on 2553 days (2018-01-01 to 2024-12-27), predicting 31 days (2025-01-01 to 2025-01-31)
In-sample MAE on the training rows: 80.3 MWh/day  (NOT a forecast accuracy)
Out-of-sample MAE, January 2025:    119.6 MWh/day


,date,actual_mwh,predicted_mwh
0,2025-01-01,3013.7,2824.1
1,2025-01-02,3404.1,3482.4
2,2025-01-03,3461.5,3428.4
3,2025-01-04,3260.4,3077.8
4,2025-01-05,3061.1,2750.1


## 6. Ridge regression

Ridge adds the L2 penalty $\alpha \sum_j \beta_j^2$ to the OLS objective. A larger $\alpha$ shrinks all coefficients more strongly towards zero. This can reduce the variance of the estimates when predictors are highly correlated, as our consumption lags are, at the cost of some bias.

**Choosing $\alpha$ without looking at the validation month.** Within each fold, $\alpha$ is chosen by time-ordered cross-validation on that fold's training rows only:
- `TimeSeriesSplit` with 5 splits: each split fits on earlier rows and scores on the period that follows, never the reverse;
- a gap of 4 days between fitting and scoring rows, mirroring the availability rule;
- score: MAE, our primary metric;
- grid: 11 values from 0.01 to 1000 on a logarithmic scale, fixed in advance. If the best $\alpha$ lies at the edge of the grid, the notebook stops, because the grid would then be too narrow.

The best $\alpha$ is then refitted on all training rows of the fold. Neither the validation month nor the 2026 holdout is ever used for tuning.

In [6]:
# Tuning settings for Ridge and Lasso (defined once, used in every fold)
RIDGE_ALPHAS = np.logspace(-2, 3, 11)        # 0.01, 0.03, 0.1, ..., 1000
LASSO_ALPHAS = np.logspace(-2, 2, 9)         # 0.01, 0.03, 0.1, ..., 100
INNER_CV = TimeSeriesSplit(n_splits=5, gap=MIN_LAG - 1)   # time-ordered, 4-day gap like the availability rule


def fit_model(model_name, spec, train):
    """Fit one model on a fold's training rows. Ridge/Lasso: alpha chosen by time-ordered CV within these rows only."""
    X, y = train[spec_columns(spec)], train[TARGET]
    pipeline = make_pipeline(model_name, spec)
    if model_name == "ols":
        return pipeline.fit(X, y), np.nan
    alphas = RIDGE_ALPHAS if model_name == "ridge" else LASSO_ALPHAS
    search = GridSearchCV(pipeline, {"model__alpha": alphas}, cv=INNER_CV, scoring="neg_mean_absolute_error")
    search.fit(X, y)   # tries every alpha, then refits the best one on all training rows of the fold
    alpha = search.best_params_["model__alpha"]
    assert alphas.min() < alpha < alphas.max(), f"{model_name}: alpha {alpha} lies at the edge of the grid"
    return search.best_estimator_, alpha


# How the inner, time-ordered cross-validation splits the January fold's training rows
for i, (inner_train, inner_valid) in enumerate(INNER_CV.split(train), start=1):
    print(f"Inner split {i}: fit {train['date'].iloc[inner_train[0]].date()} to {train['date'].iloc[inner_train[-1]].date()}"
          f" | score {train['date'].iloc[inner_valid[0]].date()} to {train['date'].iloc[inner_valid[-1]].date()}")

# Demonstration: Ridge on the January fold
ridge, ridge_alpha = fit_model("ridge", spec, train)
ridge_pred = ridge.predict(valid[cols])
print(f"\nRidge (with_weather), January 2025: alpha = {ridge_alpha:g}, "
      f"out-of-sample MAE = {compute_metrics(valid[TARGET], ridge_pred)['mae_mwh']:.1f} MWh/day")

Inner split 1: fit 2018-01-01 to 2019-02-28 | score 2019-03-05 to 2020-05-02
Inner split 2: fit 2018-01-01 to 2020-04-28 | score 2020-05-03 to 2021-07-01
Inner split 3: fit 2018-01-01 to 2021-06-27 | score 2021-07-02 to 2022-08-30
Inner split 4: fit 2018-01-01 to 2022-08-26 | score 2022-08-31 to 2023-10-29
Inner split 5: fit 2018-01-01 to 2023-10-25 | score 2023-10-30 to 2024-12-27

Ridge (with_weather), January 2025: alpha = 10, out-of-sample MAE = 109.6 MWh/day


## 7. Lasso regression

Lasso adds the L1 penalty $\alpha \sum_j |\beta_j|$. Unlike Ridge, it can set coefficients exactly to zero and thus performs a form of variable selection. $\alpha$ is tuned exactly as for Ridge, with a grid of 9 values from 0.01 to 100. Lasso is solved iteratively; we allow up to 100,000 iterations and treat any convergence warning as an error (see the setup cell).

A coefficient of zero only means that, given the other predictors and this penalty, the variable did not improve the fit in the training data. It is not evidence that the variable has no effect on electricity demand: with correlated predictors, Lasso often keeps one of several similar variables and drops the others.

In [7]:
# Demonstration: Lasso on the January fold
lasso, lasso_alpha = fit_model("lasso", spec, train)
lasso_pred = lasso.predict(valid[cols])
print(f"Lasso (with_weather), January 2025: alpha = {lasso_alpha:g}, "
      f"out-of-sample MAE = {compute_metrics(valid[TARGET], lasso_pred)['mae_mwh']:.1f} MWh/day")

lasso_coefs = pd.Series(lasso.named_steps["model"].coef_,
                        index=lasso.named_steps["preprocess"].get_feature_names_out())
zero = lasso_coefs[lasso_coefs == 0].index.tolist()
print(f"Coefficients set exactly to zero: {len(zero)} of {len(lasso_coefs)}")
print(zero)

Lasso (with_weather), January 2025: alpha = 1, out-of-sample MAE = 109.9 MWh/day
Coefficients set exactly to zero: 4 of 31
['numeric__lag_mean_5_11', 'categorical__weekday_1', 'categorical__weekday_2', 'categorical__month_6']


## 8. Monthly validation folds for 2025

For each month of 2025:

1. **Refit time:** 12:00 on the last day of the previous month, when the first forecast of the month is issued.
2. **Training rows:** all target dates from 1 January 2018 up to the training cutoff, defined as the first day of the month − 5 days (expanding window). Earlier 2025 months enter the training data once their targets are available.
3. **Fitting:** imputer, encoder, scaler and regression model are fitted on these rows only (one `Pipeline`). For Ridge and Lasso, $\alpha$ is chosen by time-ordered cross-validation within these rows (Section 6).
4. **Prediction:** every day of the validation month is predicted with this fixed model. Each day's features are available at its own forecasting cutoff (consumption lags of at least 5 days, archived weather forecasts, calendar).
5. **Storage:** each prediction is stored with its target date, model, validation month and training cutoff.

This yields 3 estimators × 2 specifications = 6 models, each with 365 out-of-sample predictions on exactly the dates of the baseline. No model ever sees actual consumption from the month it predicts. We use an expanding training window here; the comparison of expanding and rolling windows is carried out by Person 3.

In [9]:
# Overview of the 12 monthly folds: what each refit is allowed to use
fold_rows = []
for month in VALIDATION_MONTHS:
    fold_train, fold_valid, fold_cutoff = get_fold(month)
    fold_rows.append({
        "validation_month": str(month),
        "refit_at": f"{(month.start_time - pd.Timedelta(days=1)).date()} 12:00",
        "training_cutoff": fold_cutoff.date(),
        "n_train_days": len(fold_train),
        "n_validation_days": len(fold_valid),
    })
folds = pd.DataFrame(fold_rows)
display(folds)

# Checks: every 2025 day is predicted exactly once, and each cutoff respects the 5-day rule
assert folds["n_validation_days"].sum() == 365
assert folds["n_train_days"].is_monotonic_increasing   # expanding training window
feb_train, _, _ = get_fold(VALIDATION_MONTHS[1])
assert feb_train["date"].max() == pd.Timestamp("2025-01-27"), "February fold must stop at 27 January"
print("Example: the February fold uses January data up to", feb_train["date"].max().date(),
      "- the days 28–31 January are not yet available at the refit on 31 January 12:00.")

,validation_month,refit_at,training_cutoff,n_train_days,n_validation_days
0,2025-01,2024-12-31 12:00,2024-12-27,2553,31
1,2025-02,2025-01-31 12:00,2025-01-27,2584,28
2,2025-03,2025-02-28 12:00,2025-02-24,2612,31
3,2025-04,2025-03-31 12:00,2025-03-27,2643,30
4,2025-05,2025-04-30 12:00,2025-04-26,2673,31
5,2025-06,2025-05-31 12:00,2025-05-27,2704,30
6,2025-07,2025-06-30 12:00,2025-06-26,2734,31
7,2025-08,2025-07-31 12:00,2025-07-27,2765,31
8,2025-09,2025-08-31 12:00,2025-08-27,2796,30
9,2025-10,2025-09-30 12:00,2025-09-26,2826,31


Example: the February fold uses January data up to 2025-01-27 - the days 28–31 January are not yet available at the refit on 31 January 12:00.


In [10]:
# Run all 12 monthly folds: 3 estimators x 2 specifications, refitted every month
import time

MODEL_NAMES = ["ols", "ridge", "lasso"]
prediction_parts, tuning_rows, last_fold_models = [], [], {}
start_time = time.time()

for month in VALIDATION_MONTHS:
    fold_train, fold_valid, fold_cutoff = get_fold(month)
    for spec_name, fold_spec in SPECS.items():
        for model_name in MODEL_NAMES:
            label = f"{model_name}_{spec_name}"
            fitted, alpha = fit_model(model_name, fold_spec, fold_train)
            predicted = fitted.predict(fold_valid[spec_columns(fold_spec)])
            assert np.isfinite(predicted).all(), f"{label}, {month}: non-finite predictions"

            prediction_parts.append(pd.DataFrame({
                "date": fold_valid["date"].to_numpy(),
                "model": label,
                "estimator": model_name,
                "specification": spec_name,
                "validation_month": str(month),
                "training_cutoff": fold_cutoff,
                "actual_mwh": fold_valid[TARGET].to_numpy(),
                "predicted_mwh": predicted,
            }))
            tuning_rows.append({"model": label, "validation_month": str(month), "alpha": alpha,
                                "n_train_days": len(fold_train)})
            if month == VALIDATION_MONTHS[-1]:
                last_fold_models[label] = fitted   # kept for interpreting coefficients (Section 11)
    print(f"{month}: trained on {len(fold_train)} days up to {fold_cutoff.date()}, "
          f"6 models fitted ({time.time() - start_time:.0f} s elapsed)")

linear_pred = pd.concat(prediction_parts, ignore_index=True)
linear_pred["prediction_error_mwh"] = linear_pred["actual_mwh"] - linear_pred["predicted_mwh"]
linear_pred["absolute_error_mwh"] = linear_pred["prediction_error_mwh"].abs()
tuning = pd.DataFrame(tuning_rows)

# Checks: every model predicted exactly the 365 baseline dates
for label, group in linear_pred.groupby("model"):
    assert len(group) == 365 and group["date"].is_unique, f"{label}: not 365 unique dates"
    assert set(group["date"]) == set(baseline_pred["date"]), f"{label}: dates differ from the baseline"
print(f"\nDone: {linear_pred['model'].nunique()} models x 365 days = {len(linear_pred)} out-of-sample predictions")

2025-01: trained on 2553 days up to 2024-12-27, 6 models fitted (3 s elapsed)
2025-02: trained on 2584 days up to 2025-01-27, 6 models fitted (5 s elapsed)
2025-03: trained on 2612 days up to 2025-02-24, 6 models fitted (8 s elapsed)
2025-04: trained on 2643 days up to 2025-03-27, 6 models fitted (10 s elapsed)
2025-05: trained on 2673 days up to 2025-04-26, 6 models fitted (12 s elapsed)
2025-06: trained on 2704 days up to 2025-05-27, 6 models fitted (14 s elapsed)
2025-07: trained on 2734 days up to 2025-06-26, 6 models fitted (17 s elapsed)
2025-08: trained on 2765 days up to 2025-07-27, 6 models fitted (19 s elapsed)
2025-09: trained on 2796 days up to 2025-08-27, 6 models fitted (21 s elapsed)
2025-10: trained on 2826 days up to 2025-09-26, 6 models fitted (23 s elapsed)
2025-11: trained on 2857 days up to 2025-10-27, 6 models fitted (25 s elapsed)
2025-12: trained on 2887 days up to 2025-11-26, 6 models fitted (27 s elapsed)

Done: 6 models x 365 days = 2190 out-of-sample predict